<a href="https://colab.research.google.com/github/sudenazgrmz/fake-review-robustness/blob/main/notebooks/05_distilbert_fold0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import random
import time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

# ---------- reproducibility ----------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ---------- data ----------
df = pd.read_csv("deceptive-opinion.csv")
texts = df["text"].tolist()
labels = (df["deceptive"] == "deceptive").astype(int).values
groups = df["hotel"].values

folds = list(GroupKFold(n_splits=5).split(texts, labels, groups))
train_idx, test_idx = folds[0]
print("train size:", len(train_idx), "test size:", len(test_idx))
print("test hotels:", sorted(set(groups[test_idx])))

X_train = [texts[i] for i in train_idx]
X_test = [texts[i] for i in test_idx]
y_train = labels[train_idx]
y_test = labels[test_idx]

# ---------- TF-IDF baselines on the SAME fold ----------
for name, clf in [("tfidf_logreg", LogisticRegression(max_iter=1000)),
                  ("tfidf_svm", LinearSVC())]:
    pipe = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2), clf)
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    print(name, "fold-0 macro-F1:", round(f1_score(y_test, pred, average="macro"), 3))

# ---------- DistilBERT ----------
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 256
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5
device = "cuda"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(text_list):
    enc = tokenizer(text_list, truncation=True, max_length=MAX_LEN,
                    padding="max_length", return_tensors="pt")
    return enc["input_ids"], enc["attention_mask"]

train_ids, train_mask = encode(X_train)
test_ids, test_mask = encode(X_test)

train_ds = TensorDataset(train_ids, train_mask, torch.tensor(y_train))
test_ds = TensorDataset(test_ids, test_mask, torch.tensor(y_test))

gen = torch.Generator()
gen.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=gen)
test_loader = DataLoader(test_ds, batch_size=64)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

start = time.time()
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for ids, mask, lab in train_loader:
        ids, mask, lab = ids.to(device), mask.to(device), lab.to(device)
        out = model(input_ids=ids, attention_mask=mask, labels=lab)
        out.loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += out.loss.item()
    print("epoch", epoch + 1, "avg train loss:", round(total_loss / len(train_loader), 4))
print("training time (s):", round(time.time() - start))

# ---------- evaluate ONCE, after training ----------
model.eval()
preds = []
with torch.no_grad():
    for ids, mask, lab in test_loader:
        out = model(input_ids=ids.to(device), attention_mask=mask.to(device))
        preds.extend(out.logits.argmax(dim=-1).cpu().tolist())

print()
print("distilbert fold-0 macro-F1:", round(f1_score(y_test, preds, average="macro"), 3))
print(classification_report(y_test, preds, target_names=["truthful", "deceptive"]))

train size: 1280 test size: 320
test hotels: ['conrad', 'hyatt', 'omni', 'talbott']
tfidf_logreg fold-0 macro-F1: 0.862
tfidf_svm fold-0 macro-F1: 0.872


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1 avg train loss: 0.5411
epoch 2 avg train loss: 0.2685
epoch 3 avg train loss: 0.1409
training time (s): 78

distilbert fold-0 macro-F1: 0.874
              precision    recall  f1-score   support

    truthful       0.94      0.80      0.86       160
   deceptive       0.83      0.95      0.88       160

    accuracy                           0.88       320
   macro avg       0.88      0.88      0.87       320
weighted avg       0.88      0.88      0.87       320

